# Imports

In [1]:
# Local application/library specific imports
from pygrex.config import cfg
from pygrex.data_reader import DataReader, GroupInteractionHandler
from pygrex.evaluator import run_evaluation_with_proper_split
from pygrex.explain.groups.lore4groups_explainer import LORE4GroupsExplainer
from pygrex.models import ALS
from pygrex.recommender import GroupRecommender
from pygrex.utils import AggregationStrategy
from pygrex.evaluator import ExplanationEvaluator

import time
import pandas as pd
import os


In [2]:
# Read the ratings file.
data = DataReader(**cfg.data.test)
data.make_consecutive_ids_in_dataset()
# data.binarize(binary_threshold=1)

# Read the file with the group ids
group_handler = GroupInteractionHandler(**cfg.data.groups)
available_groups = group_handler.read_groups("groupsWithHighRatings5.txt")
print("✅ Data preparation complete.\n")

# --- Display Data Summary ---
print("--- Data Summary ---")
print(f"👥 Unique Users: {data.num_user:,}")
print(f"📦 Unique Items: {data.num_item:,}")
print(f"⭐ Total Ratings: {len(data.get_raw_dataset()):,}")
print(f"👨‍👩‍👧‍👦 Number of Groups: {len(available_groups):,}")
print("\nProcessed Ratings DataFrame Head:")
display(data.dataset.head())

✅ Data preparation complete.

--- Data Summary ---
👥 Unique Users: 610
📦 Unique Items: 9,724
⭐ Total Ratings: 100,836
👨‍👩‍👧‍👦 Number of Groups: 17

Processed Ratings DataFrame Head:


,userId,itemId,rating,timestamp
0,0,0,4.0,964982703
1,0,2,4.0,964981247
2,0,5,4.0,964982224
3,0,43,5.0,964983815
4,0,46,5.0,964982931


## Step 2: Model Training & Evaluation

With the data prepared, we now select and train a recommendation model. We will use **Alternating Least Squares (ALS)**, a matrix factorization technique for implicit feedback. After training, we will evaluate its performance using a train/test split to measure its Hit Ratio and NDCG.

In [3]:
print("--- 2.1 Model Training ---")

# Train the recommendation model
model = ALS(**cfg.model.als)

# Train the model
start_time = time.time()
model.fit(data)
end_time = time.time()
training_time = end_time - start_time

print(f"✅ Model trained successfully in {training_time:.2f} seconds!")

--- 2.1 Model Training ---


c:\Users\usuar\miniconda3\envs\pygrex-exp-grs\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/10 [00:00<?, ?it/s]

✅ Model trained successfully in 0.95 seconds!


In [ ]:
print("\n--- 2.2 Offline Model Evaluation ---")
# For evaluation, a new model instance must be created.
# The evaluation function handles its own internal data splitting and training.
eval_model = ALS(**cfg.model.als)

# Define evaluation parameters
test_size = 0.2
top_n = 10

print(f"Running evaluation with a {test_size*100:.0f}% test split (Top-{top_n})...")

# Run the evaluation
evaluation_scores = run_evaluation_with_proper_split(
    data_reader=data,
    model=eval_model,
    test_size=test_size,
    top_n=top_n,
)

# Display evaluation results
print("\n--- Evaluation Results ---")
print(f"Hit Ratio @{top_n}: {evaluation_scores.get('Hit Ratio', 0.0):.2%}")
print(f"NDCG @{top_n}: {evaluation_scores.get('NDCG', 0.0):.4f}")
print(f"Evaluation Time: {evaluation_scores.get('evaluation_time', 0):.1f}s")

## Step 3: Group Recommendation

Now that we have a trained model, we can generate recommendations for a group. We will select a group, choose an aggregation strategy to combine individual member preferences, and generate a Top-10 list of recommended items.

In [4]:
print("--- 3. Group Recommendation ---")

# Select a group and strategy
selected_group_id = available_groups[0]  # Let's use the first group as an example
group_members = group_handler.parse_group_members(selected_group_id)
aggregation_strategy = AggregationStrategy.AVG_PREDICTIONS # Use the simple average strategy
top_k = 10

print(f"Generating Top-{top_k} recommendations for group: {selected_group_id}")
print(f"👥 Group Members: {group_members}")
print(f"📊 Aggregation Strategy: {aggregation_strategy.name}")

# --- Generate Recommendations ---
# 1. Instantiate the GroupRecommender
group_recommender = GroupRecommender(data=data)

# 2. Setup the recommendation process
group_recommender.setup_recommendation(
    model=model,
    members=group_members,  # type: ignore
    data=data,
    aggregation_strategy=aggregation_strategy,
                )


# 3. Get the final recommendation list
recommended_items = group_recommender.get_group_recommendations(top_k=top_k)
recommendation_scores = group_recommender.get_recommendation_scores()

print("\n✅ Recommendations generated successfully!")

# --- Display Results ---
rec_data = [
    {
        "Rank": i + 1,
        "Item ID": item_id,
        "Aggregated Score": recommendation_scores.get(item_id, 0.0),
    }
    for i, item_id in enumerate(recommended_items)  # type: ignore
]

rec_df = pd.DataFrame(rec_data)
print(f"\nTop {top_k} Recommended Items:")
display(rec_df)

--- 3. Group Recommendation ---
Generating Top-10 recommendations for group: 522_385_234_452_594
👥 Group Members: [522, 385, 234, 452, 594]
📊 Aggregation Strategy: AVG_PREDICTIONS

✅ Recommendations generated successfully!

Top 10 Recommended Items:


,Rank,Item ID,Aggregated Score
0,1,543,4.636274
1,2,757,4.582981
2,3,564,4.504107
3,4,441,4.488708
4,5,379,4.341830
5,6,475,4.279482
6,7,43,4.268454
7,8,19,4.225248
8,9,748,4.178329
9,10,64,4.147735


## Step 4: Explanation (LORE4Groups)

Finally, we generate an explanation for the recommended items using **LORE4Groups**, a local rule-based method. It:
- builds a local neighborhood of similar items using tag profiles
- trains a simple decision tree per item to predict 'like' vs 'not like'
- extracts interpretable rules explaining why items were recommended.


In [5]:
print("--- 4. Local Rule-Based Explanation (LORE4Groups) ---")

# 1) Build tag-based item profiles aligned with the ratings dataset
# ---------------------------------------------------------------
# Read tags file from config
_tags_path = cfg.data.tags.tags_file
if not os.path.exists(_tags_path):
    raise SystemExit(f"Tags file not found at: {_tags_path}")

# Read tags and align original -> consecutive item ids
tags_df = pd.read_csv(_tags_path)
consecutive_items = set(data.dataset["itemId"].unique())
original_to_consecutive = {}
for item_consec in consecutive_items:
    try:
        item_orig = data.get_original_item_id(int(item_consec))
        original_to_consecutive[item_orig] = int(item_consec)
    except (ValueError, KeyError):
        continue

# Keep only tags for items present in ratings
tags_df = tags_df[tags_df["movieId"].isin(original_to_consecutive.keys())].copy()
if len(tags_df) == 0:
    raise SystemExit("No tag data matches items in ratings dataset.")

# Normalize labels (keep full label as tag, lowercase)
tags_df["label"] = tags_df["label"].astype(str).str.lower().str.strip()
# Map to consecutive ids
tags_df["movieId"] = tags_df["movieId"].map(original_to_consecutive).astype(int)

# Keep top-N most frequent labels to reduce sparsity
_top_n = cfg.explainer.lore4groups.top_n_labels
top_labels = (
    tags_df["label"].value_counts().nlargest(_top_n).index.tolist()
)
tags_final = tags_df[tags_df["label"].isin(top_labels)].copy()

# Item profiles: {str(itemId): set(labels)}
item_profiles = (
    tags_final.groupby("movieId")["label"].apply(set).to_dict()
)
item_profiles = {str(k): v for k, v in item_profiles.items()}

# Item-label matrix (rows: itemId as str, cols: labels, values: 0/1)
item_label_matrix = tags_final.assign(value=1).pivot_table(
    index="movieId", columns="label", values="value", fill_value=0
)
item_label_matrix.index = item_label_matrix.index.astype(str)

# 2) Prepare user history in required format
# ------------------------------------------
user_hist = {}
for user_id_orig in group_members:
    try:
        user_id_consec = data.get_new_user_id(user_id_orig)
        hist_items = set(
            data.dataset[data.dataset["userId"] == user_id_consec]["itemId"].astype(str)
        )
        user_hist[user_id_orig] = hist_items
    except Exception:
        user_hist[user_id_orig] = set()

# Filter recommendations to those we can explain (must exist in profiles)
explainable_recs = [str(i) for i in recommended_items if str(i) in item_profiles]
if not explainable_recs:
    print("⚠️ No recommended items have sufficient tag data for explanation.")
else:
    # 3) Run LORE4Groups explainer
    explainer = LORE4GroupsExplainer(
        item_profiles=item_profiles,
        item_label_matrix=item_label_matrix,
        config=cfg,
        genre_profiles=None,  # optional, omitted for toy example
    )

    results = explainer.find_explanation(
        explainable_recs,
        group_members,
        user_hist,
        data.dataset,
        model=model,
        data_reader=data,
    )

    fidelity = results.get("fidelity", 0.0)
    details = results.get("details", {})

    print("Explanation Fidelity:")
    print(f"{fidelity:.2%}")
    print("-" * 20)

    # Compute GILD diversity like in the app
    evaluator = ExplanationEvaluator()
    metrics = evaluator.evaluate({"fidelity": fidelity, "details": details}, explainer_type="LORE4Groups")
    print("Explanation Diversity (GILD):")
    print(f"{metrics.get('gild', 0.0):.4f}")
    print("-" * 20)

    print("Items with explanations:")
    print(list(details.keys()))
    print("-" * 20)

    # Optionally preview one item's explanation summary if available
    if details:
        first_item, exp = next(iter(details.items()))
        decision_path = exp.get("decision_path", [])
        group_factual = exp.get("group_factual_rule", [])
        print(f"Sample item: {first_item}")
        print("Decision Path (rules):", decision_path)
        print("Group Factual Rules:", group_factual)


--- 4. Local Rule-Based Explanation (LORE4Groups) ---
Explanation Fidelity:
25.00%
--------------------
Explanation Diversity (GILD):
0.8871
--------------------
Items with explanations:
['475', '43']
--------------------
Sample item: 475
Decision Path (rules): ['nudity (rear) <= 0.50', 'twins <= 0.50']
Group Factual Rules: {'unanimous': [], 'majority': [], 'minority': ['70mm <= 0.50 (1/5 members)', 'franchise <= 0.50 (1/5 members)', 'futuristmoviescom <= 0.50 (1/5 members)', 'nudity (rear) <= 0.50 (1/5 members)', 'owned <= 0.50 (1/5 members)', 'seen at the cinema <= 0.50 (1/5 members)', 'sequel <= 0.50 (1/5 members)', 'twins <= 0.50 (1/5 members)']}
